# Imports

In [26]:
# import muon
import numpy as np
import mudata as md
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import pandas as pd

# clustering
from sklearn.cluster import KMeans

# Read Data

Working off of the preprocessed tonsil h5mu

In [2]:
mdata = md.read_h5mu("../../data/tonsil/tonsil_pp.h5mu")

/opt/anaconda3/lib/python3.13/site-packages/mudata/_core/mudata.py:1598: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/opt/anaconda3/lib/python3.13/site-packages/mudata/_core/mudata.py:1461: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


In [4]:
mdata

MuData object with n_obs × n_vars = 2492 × 2283
  2 modalities
    RNA:	2492 x 2000
      obs:	'leiden'
      uns:	'leiden', 'leiden_colors', 'neighbors', 'pca', 'umap'
      obsm:	'X_pca', 'X_umap', 'pos'
      varm:	'PCs'
      obsp:	'connectivities', 'distances'
    Protein:	2492 x 283
      obs:	'leiden'
      uns:	'leiden', 'leiden_colors', 'neighbors', 'pca', 'umap'
      obsm:	'X_pca', 'X_umap', 'pos'
      varm:	'PCs'
      obsp:	'connectivities', 'distances'

Presuming the latent variables are somewhere in the MuData object, then I'll just call the layer accordingly.

# Clustering Metrics

In [ ]:
# read the embeddings file
embeddings = pd.read_table("embeddings.txt", header = 0)

## given that embeddings will be an m x k matrix, where m = cells and k = latent dimensions, maybe can pull the mdata object and add the embeddings into obsm layer
rna_adata = mdata.mod['RNA']
rna_adata.obsm['spatial_totalVI_embedding'] = embeddings
sc.pp.neighbors(rna_adata, use_rep="spatial_totalVI_embedding", n_neighbors=30, metric="correlation") # it SHOULD make the umap based on this neighbor's adjacency right?
sc.tl.umap(rna_adata, min_dist=0.4)

## K-means

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=0).fit(embeddings) # not sure how many clusters to create?
rna_adata.obs['kmeans5'] = kmeans.labels_.astype(str)

## Leiden

In [ ]:
sc.tl.leiden(rna_adata, key_added="leiden_eval", resolution=0.7)

## Louvain

In [ ]:
sc.tl.louvain(rna_adata, key_added="leiden_eval", resolution=0.7)

ModuleNotFoundError: No module named 'louvain'

# UMAP

In [ ]:
sc.pl.umap(rna_adata, color = ['kmeans5', 'louvain_eval', 'leiden_eval'])